In [1]:
import torch
import torch.nn as nn

In [2]:
# Model
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.net(x)

In [3]:
# define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
#转移模型
model = MLP().to(device)

In [5]:
#loss & optimizer组件
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [6]:
# dataset & loader
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# transform流水线---转化原始数据

train_data = datasets.FashionMNIST(
    root="data",      # 数据下载到哪个目录
    train=True,       # True=训练集，False=测试集
    download=False,    # 本地没有就下载
    transform=transform # 绑定刚刚定义的流水线，取样本
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=False,
    transform=transform
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)


In [7]:
# 训练循环
def run_epoch(model, loader, criterion, optimizer, device):
    total_loss = 0
    model.train() #切换训练模式
    for batch_imgs, batch_labels in loader:
        
        #转移数据
        batch_imgs = batch_imgs.to(device)
        batch_labels = batch_labels.to(device)
        
        #forward
        logits = model(batch_imgs)
        loss = criterion(logits, batch_labels) # softmax & NLL 得到 loss --- 注意这里的loss已经是batch内部的均值
        
        #backward
        optimizer.zero_grad() #清零梯度
        loss.backward()
        optimizer.step()

        total_loss += loss.item() #转换loss数据类型为float

    return total_loss / len(loader)   


In [8]:
# 外层循环
num_epochs = 10
train_losses = []
for epoch in range(num_epochs):
    avg_loss = run_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f} ")

Epoch 1/10, Loss: 0.5207 
Epoch 2/10, Loss: 0.3788 
Epoch 3/10, Loss: 0.3367 
Epoch 4/10, Loss: 0.3146 
Epoch 5/10, Loss: 0.2949 
Epoch 6/10, Loss: 0.2788 
Epoch 7/10, Loss: 0.2680 
Epoch 8/10, Loss: 0.2541 
Epoch 9/10, Loss: 0.2437 
Epoch 10/10, Loss: 0.2342 


In [9]:
# 测试function
def evaluate(model, loader, device):
    model.eval() # 切换推理模式
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_imgs, batch_labels in loader:  
            #转移数据
            batch_imgs = batch_imgs.to(device)
            batch_labels = batch_labels.to(device)
            
            logits = model(batch_imgs) #计算原始输出 (64,10)
            predictions = torch.argmax(logits, dim = 1) #从第一维 "10"开始
            #预测的类别就是p最大的类别---用argmax找
            
            correct += (predictions == batch_labels).sum().item() #预测成功的数量
            total += batch_labels.size(0) #只要第0维长度 同.shape
    return correct / total
test_accuracy = evaluate(model, test_loader, device)
print(f"TEST_ACCURACY : {test_accuracy:.4f}")

TEST_ACCURACY : 0.8828
